
# DistilBERT (PyTorch) Fine-Tuning — BBC Text (Multi‑Class)

This notebook fine‑tunes **DistilBERT** using **PyTorch + 🤗 Transformers** on the attached **BBC Text** dataset (`/mnt/data/bbc-text.csv`).  
It includes installs, preprocessing, training with `Trainer`, evaluation, and an inference helper.

**Labels:** 5 classes (business, entertainment, politics, sport, tech)  
**Baseline model:** `distilbert-base-uncased`

> If you run this on GPU, keep an eye on VRAM; reduce `per_device_train_batch_size` if you hit OOM.


In [ ]:

# 🔧 Installs (rerun if environment changes). For CUDA 12.x wheels, use the CUDA index URL.
# If you're on CPU only, remove the index-url line.
import sys, subprocess, pkgutil

def pip_install(args):
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + args)

# Core packages
to_install = [
    "torch==2.4.*",
    "--index-url", "https://download.pytorch.org/whl/cu121",
]
pip_install(to_install)

pip_install([
    "transformers==4.44.2",
    "datasets>=2.20",
    "evaluate",
    "accelerate>=0.34.0",
    "pandas",
    "scikit-learn",
    "safetensors>=0.4.4"
])

import torch, transformers, datasets, evaluate, pandas as pd
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())


## Load & Inspect Dataset

In [ ]:

import pandas as pd
from pathlib import Path

DATA_PATH = Path("/mnt/data/bbc-text.csv")

df = pd.read_csv(DATA_PATH)
print(df.head(3))
print("Rows:", len(df), "Columns:", list(df.columns))

# Expected columns: 'category', 'text'
assert {'category','text'}.issubset(df.columns), "Expected columns 'category' and 'text' in the CSV."


## Encode Labels

In [ ]:

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label'] = le.fit_transform(df['category'].astype(str))

id2label = {i: lab for i, lab in enumerate(le.classes_)}
label2id = {v: k for k, v in id2label.items()}

print("Classes:", id2label)
df[['category','label']].head()


## Train / Validation / Test Split

In [ ]:

from sklearn.model_selection import train_test_split

data_texts = df['text'].astype(str).tolist()
data_labels = df['label'].tolist()

# 80/20 train-val split, then carve out 1% of train as test (tiny sanity test)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    data_texts, data_labels, test_size=0.2, random_state=0, stratify=data_labels
)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    train_texts, train_labels, test_size=0.01, random_state=0, stratify=train_labels
)

len(train_texts), len(val_texts), len(test_texts)


## Tokenize with DistilBERT Tokenizer

In [ ]:

from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch_texts):
    return tokenizer(batch_texts, truncation=True, padding=False)

train_enc = tokenize_function(train_texts)
val_enc   = tokenize_function(val_texts)
test_enc  = tokenize_function(test_texts)

# Quick check
{k: (type(v), len(v), (len(v[0]) if hasattr(v[0],'__len__') else None)) for k,v in train_enc.items()}


## Build PyTorch Datasets

In [ ]:

import torch

class HFDictDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = HFDictDataset(train_enc, train_labels)
val_ds   = HFDictDataset(val_enc, val_labels)
test_ds  = HFDictDataset(test_enc, test_labels)

from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


## Model, TrainingArguments, and Metrics

In [ ]:

import numpy as np
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import evaluate

NUM_LABELS = len(id2label)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_micro": f1_metric.compute(predictions=preds, references=labels, average="micro")["f1"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

training_args = TrainingArguments(
    output_dir="distilbert-pytorch-bbc",
    learning_rate=5e-5,
    per_device_train_batch_size=16,   # adjust for your GPU memory
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)


## Train

In [ ]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
train_result.metrics


## Evaluate on Validation & Test

In [ ]:

from sklearn.metrics import accuracy_score, f1_score, classification_report

val_metrics = trainer.evaluate()
print("Validation metrics:", val_metrics)

pred = trainer.predict(test_ds)
test_preds = np.argmax(pred.predictions, axis=-1)
print("Test accuracy:", accuracy_score(test_labels, test_preds))
print("Test micro-F1:", f1_score(test_labels, test_preds, average="micro"))
print("Test macro-F1:", f1_score(test_labels, test_preds, average="macro"))
print("\nClassification report on Test:")
print(classification_report(test_labels, test_preds, target_names=[id2label[i] for i in range(NUM_LABELS)]))


## Save Best Model & Tokenizer

In [ ]:

save_dir = "distilbert-pytorch-bbc/best"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved to:", save_dir)


## Inference Helper

In [ ]:

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

def load_saved_model(path="distilbert-pytorch-bbc/best"):
    tok = AutoTokenizer.from_pretrained(path)
    mdl = AutoModelForSequenceClassification.from_pretrained(path)
    mdl.eval()
    return tok, mdl

def predict(texts, path="distilbert-pytorch-bbc/best"):
    tok, mdl = load_saved_model(path)
    batch = tok(texts, truncation=True, padding=True, return_tensors="pt")
    with torch.no_grad():
        logits = mdl(**batch).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        pred_ids = probs.argmax(axis=-1)
    pred_labels = [mdl.config.id2label[i] for i in pred_ids]
    return pred_labels, probs

samples = [
    "The central bank raised interest rates amid concerns over inflation.",
    "The team clinched the championship in a thrilling final match.",
    "The new smartphone features an improved camera and faster processor."
]

labels, probabilities = predict(samples)
for s, l in zip(samples, labels):
    print(f"[{l}] {s}")



## Troubleshooting

- **CUDA OOM:** Reduce `per_device_train_batch_size` to 8 or 4.
- **Slow training on CPU:** Consider running on GPU; keep `fp16=True` (default when CUDA is available).
- **Version mismatch errors:** Re-run the install cell; ensure `transformers==4.44.2`, `torch==2.4.*`.
- **Label errors:** Ensure the CSV has `category` (string) and `text` columns.
